# ViSP — Full fine-tune trên dữ liệu đã review

**Bộ dữ liệu gồm 1.050 câu đã được nhóm review**

697 cặp train, 141 dev, 150 test sau khi tách pilot và nguồn mơ hồ. Cập nhật toàn bộ trọng số ViSP, không dùng LoRA. JFLEG/SEEDA là cơ sở đánh giá, không phải hai model được train.

Notebook chứa corpus riêng tư của nhóm, không có credentials database. Bật GPU T4 và Internet. Không công bố notebook/dữ liệu khi chưa kiểm tra quyền nguồn. Các bước train/test phía dưới thực thi thật, không dùng số liệu giả định.

## 1. Khôi phục mã nguồn và dữ liệu đã khóa

In [ ]:
from pathlib import Path
import os,sys,subprocess,zipfile,io,base64,hashlib,json
os.environ['USE_TF']='0'
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='66dba39a55e559b2b57ba405420d3dea0ebca544bccaa0af650706c8dc626f26'
work=Path('/kaggle/working/visp_full') if Path('/kaggle/working').exists() else Path.cwd()/'notebook_work'
work.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(work)
print('Workspace:',work)
def run(*args):
    process=subprocess.Popen([sys.executable,'-u','-X','utf8',*args],cwd=work,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout: print(line,end='',flush=True)
    if process.wait(): raise RuntimeError('Command failed: '+' '.join(args))
run('-m','pip','install','--quiet','-r','requirements.txt')
print(json.loads((work/'data/manifest.json').read_text(encoding='utf-8'))['counts'])


## 2. Kiểm tra phần cứng và smoke test
Smoke test tiny T5 kiểm tra optimizer và resume; không phải kết quả ViSP.

In [ ]:
import torch
print('Torch:',torch.__version__)
assert torch.cuda.is_available(), 'Bật Kaggle GPU trước khi training'
print(torch.cuda.get_device_name(0))
run('-m','unittest','-v','test_finetune')

## 3. Full fine-tune
Hai learning rate, chọn checkpoint trên dev. Batch giảm 2→1 nếu OOM; effective batch giữ 16. Checkpoint resume ở cuối mỗi epoch.

In [ ]:
run('full_finetune.py','train','--data','data','--output','runs')

## 4. So ViSP gốc và ViSP sau fine-tune
Cùng beam search 5, không sampling. Chỉ mở test sau khi khóa checkpoint bằng dev.

In [ ]:
run('full_finetune.py','evaluate','--data','data','--output','runs')
print((work/'runs/test_metrics.json').read_text(encoding='utf-8'))

## 5. Log và checkpoint
Model đã train nằm trong `runs/lr_*/best`; xem `selected_model.json` để biết model được chọn. Chưa có judgments người đọc thì các metric harm/fluency để pending.

In [ ]:
import matplotlib.pyplot as plt
selection=json.loads((work/'runs/selected_model.json').read_text(encoding='utf-8'))
for result in selection['runs']:
    history=result['history']
    plt.plot([r['epoch'] for r in history],[r['dev']['balanced'] for r in history],label=str(result['lr']))
plt.xlabel('Epoch');plt.ylabel('Balanced dev token loss');plt.legend();plt.show()
print('Chosen model:',selection['winner']['model_dir'])
print('Reload verification:',(work/'runs/reload_verification.json').read_text())